# Tomahawk 16 T — power-basis reconciliation

Attach the **same original** `Tomahawk_16T_Full_MGRID_MHD_Power_Input.zip` used in the three completed field runs as **one Kaggle input**. It can be mounted as a ZIP or extracted files, then select **Save Version → Save & Run All**. No field recomputation occurs; this runs in seconds.

The notebook verifies all source files against their SHA-256 manifest and checks that the manifest is the one recorded in the completed 96-plane MGRID runs. It uses the CSV pressure to establish whether `density_m3` is total ion density. It then computes the current source, an earlier selected 20 keV design profile, the model builder's 30 keV profile, and a separate density sensitivity at the current temperature shape. All figures use the same explicit plant assumptions for a direct comparison, without an added 15.08 GWe anchor offset.

The results appear in `/kaggle/working/Tomahawk_16T_Power_Reconciliation_Results.zip` and in separate CSV/JSON files. The higher-density case is a sensitivity, not an achieved operating state. This notebook does not change the finished vacuum MGRID or silently relabel the original pressure/density profile.


In [ ]:
import importlib
import subprocess
import sys
try:
    importlib.import_module("numpy")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
import numpy as np
print("Python", sys.version.split()[0], "NumPy", np.__version__)


In [ ]:
"""Reconcile the Tomahawk 16 T source profile with the earlier power basis.

Runs on Kaggle with the original 16 T source package attached as an input.
The MGRID field data are unaffected and are not recalculated.
"""

import csv
import hashlib
import io
import json
import os
import zipfile
from pathlib import Path

import numpy as np

EXPECTED_MANIFEST_SHA256 = "797b100de7e11ec2a3af4d29ee25415a46951772bcc6f4ab5bb141e0a556a2a7"
SOURCE_ZIP_NAME = "Tomahawk_16T_Full_MGRID_MHD_Power_Input.zip"
KEV_TO_J = 1.602176634e-16
FUSION_ENERGY_J = 17.6e6 * 1.602176634e-19
VOLUME_M3 = 950.0
ACTIVE_CORES = 10
ETA = 0.40
RECIRC_GWE = 3.0
CAPTURE_NEUTRON = 0.95
DEPOSIT_ALPHA = 0.90


def locate_source():
    configured = os.environ.get("TOMAHAWK_SOURCE_ZIP")
    if configured:
        candidate = Path(configured)
        if not candidate.exists():
            raise FileNotFoundError(f"TOMAHAWK_SOURCE_ZIP does not exist: {candidate}")
        return candidate
    for root in (Path("/kaggle/input"), Path.cwd()):
        if root.is_dir():
            for manifest_path in sorted(root.rglob("MANIFEST_SHA256.json")):
                if hashlib.sha256(manifest_path.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA256:
                    return manifest_path.parent
            for path in sorted(root.rglob("*.zip")):
                try:
                    with zipfile.ZipFile(path) as archive:
                        for name in archive.namelist():
                            if (name.endswith("MANIFEST_SHA256.json") and
                                    hashlib.sha256(archive.read(name)).hexdigest() == EXPECTED_MANIFEST_SHA256):
                                return path
                except (OSError, zipfile.BadZipFile):
                    continue
    mounted = Path("/kaggle/input")
    available = sorted(p.name for p in mounted.iterdir()) if mounted.is_dir() else []
    raise FileNotFoundError(
        f"Original 16 T source package not found. Kaggle input datasets: {available}. "
        f"Attach the dataset containing {SOURCE_ZIP_NAME} or its extracted files."
    )


def load_and_check_source(path):
    def verify_and_parse(read_member):
        manifest_bytes = read_member("MANIFEST_SHA256.json")
        manifest_hash = hashlib.sha256(manifest_bytes).hexdigest()
        if manifest_hash != EXPECTED_MANIFEST_SHA256:
            raise ValueError("Source manifest differs from the three finished MGRID runs")
        manifest = json.loads(manifest_bytes)
        for member, expected in manifest.items():
            try:
                actual = hashlib.sha256(read_member(member)).hexdigest()
            except (FileNotFoundError, KeyError) as exc:
                raise ValueError(f"Missing source member: {member}") from exc
            if actual != expected:
                raise ValueError(f"Missing or modified source member: {member}")
        profile_name = "MHD/mhd_profile_sweep.csv"
        table_name = "MHD/bosch_hale_uw_table6.csv"
        rows = list(csv.DictReader(io.StringIO(read_member(profile_name).decode("utf-8-sig"))))
        table = list(csv.DictReader(io.StringIO(read_member(table_name).decode("utf-8-sig"))))
        return manifest_hash, rows, table

    if path.is_dir():
        return verify_and_parse(lambda member: (path / member).read_bytes())
    with zipfile.ZipFile(path) as archive:
        if archive.testzip() is not None:
            raise ValueError("The source ZIP contains a damaged file")
        names = [name for name in archive.namelist() if name.endswith("MANIFEST_SHA256.json")]
        matches = [name for name in names if hashlib.sha256(archive.read(name)).hexdigest()
                   == EXPECTED_MANIFEST_SHA256]
        if len(matches) != 1:
            raise ValueError("Source ZIP must contain one matching MANIFEST_SHA256.json")
        prefix = matches[0].removesuffix("MANIFEST_SHA256.json")
        return verify_and_parse(lambda member: archive.read(prefix + member))


def dt_reactivity(t_keV):
    t = np.asarray(t_keV, dtype=float)
    if not np.isfinite(t).all() or np.any(t < 0.2) or np.any(t > 100):
        raise ValueError("Bosch-Hale D-T fit requires 0.2 <= Ti <= 100 keV")
    bg, mrc2 = 34.3827, 1.124656e6
    c1, c2, c3, c4 = 1.17302e-9, 1.51361e-2, 7.51886e-2, 4.60643e-3
    c5, c6, c7 = 1.35000e-2, -1.06750e-4, 1.36600e-5
    num = t * (c2 + t * (c4 + t * c6))
    den = 1.0 + t * (c3 + t * (c5 + t * c7))
    theta = t / (1.0 - num / den)
    xi = (bg**2 / (4.0 * theta)) ** (1.0 / 3.0)
    return 1e-6 * c1 * theta * np.sqrt(xi / (mrc2 * t**3)) * np.exp(-3.0 * xi)


def radial_fusion_source(n_d, n_t, t_i, rho, weights):
    return float(VOLUME_M3 * np.sum(weights * 2.0 * rho * n_d * n_t *
                                     dt_reactivity(t_i) * FUSION_ENERGY_J) / 1e9)


def conditional_net(source_gw_core):
    usable = (14.1 / 17.6 * CAPTURE_NEUTRON + 3.5 / 17.6 * DEPOSIT_ALPHA)
    return ACTIVE_CORES * source_gw_core * usable * ETA - RECIRC_GWE


def main():
    source_path = locate_source()
    print("Source package:", source_path)
    manifest_hash, profile_rows, reference = load_and_check_source(source_path)
    xref = np.array([float(row["T_keV"]) for row in reference])
    yref = np.array([float(row["reactivity_cm3_s"]) * 1e-6 for row in reference])
    reference_error = float(np.max(np.abs(dt_reactivity(xref) / yref - 1)))
    if reference_error >= 0.005:
        raise ValueError("D-T reactivity does not agree with the source reference table")

    psi = np.array([float(row["normalized_flux"]) for row in profile_rows])
    density = np.array([float(row["density_m3"]) for row in profile_rows])
    ti = np.array([float(row["ion_temperature_keV"]) for row in profile_rows])
    te = np.array([float(row["electron_temperature_keV"]) for row in profile_rows])
    pressure = np.array([float(row["pressure_Pa"]) for row in profile_rows])
    if (not np.all(np.diff(psi) > 0) or not np.isclose(psi[0], 0)
            or not np.isclose(psi[-1], 1) or np.any(density <= 0)):
        raise ValueError("Source profile grid or density is invalid")
    # For fully ionized, equal D/T and no impurity: n_i = n_e = n_D + n_T.
    pressure_total = density * (ti + te) * KEV_TO_J
    pressure_error = float(np.max(np.abs(pressure_total / pressure - 1)))
    if pressure_error > 1e-4:
        raise ValueError("Density convention is not established by the pressure column")

    q, w = np.polynomial.legendre.leggauss(64)
    rho, weights = (q + 1.0) / 2.0, w / 2.0
    current_n_total = np.interp(rho**2, psi, density)
    current_ti = np.interp(rho**2, psi, ti)
    source_current = radial_fusion_source(current_n_total / 2, current_n_total / 2,
                                          current_ti, rho, weights)
    # Restore the earlier species-center density while retaining the supplied 22 keV shape.
    density_scale = (2 * 1.2e20) / density[0]
    source_rebased = source_current * density_scale**2

    q_old, w_old = np.polynomial.legendre.leggauss(256)
    rho_old, weights_old = (q_old + 1.0) / 2.0, w_old / 2.0
    species_selected = 1.2e20 * 1.06 * (1.0 - rho_old**2)**0.657
    ti_selected = 1.25 + (20.0 - 1.25) * (1.0 - rho_old**2)**0.75
    source_selected = radial_fusion_source(species_selected, species_selected,
                                           ti_selected, rho_old, weights_old)
    species_old30 = 1.2e20 * (1.0 - rho_old**2)**0.657
    ti_old30 = 1.25 + (30.0 - 1.25) * (1.0 - rho_old**2)**0.80
    source_old30 = radial_fusion_source(species_old30, species_old30,
                                        ti_old30, rho_old, weights_old)

    rows = []
    def add(label, src, n_total_center, ti_center, basis):
        rows.append({
            "scenario": label,
            "central_total_ion_density_m3": n_total_center,
            "central_ion_temperature_keV": ti_center,
            "fusion_source_GWth_per_core": round(src, 9),
            "captured_fraction_assumed": round(14.1 / 17.6 * CAPTURE_NEUTRON +
                                               3.5 / 17.6 * DEPOSIT_ALPHA, 9),
            "active_cores": ACTIVE_CORES,
            "cycle_efficiency_assumed": ETA,
            "recirculating_GWe_assumed": RECIRC_GWE,
            "conditional_net_GWe_no_anchor_offset": round(conditional_net(src), 9),
            "basis": basis,
        })
    add("current_source_CSV", source_current, density[0], ti[0],
        "Original total-ion density; source ZIP and MGRID run input")
    add("same_T_profile_earlier_species_center_density", source_rebased,
        density[0] * density_scale, ti[0],
        "Sensitivity only: pressure and density must both increase by density scale")
    add("earlier_selected_20keV_profile_replay", source_selected,
        2 * 1.2e20 * 1.06, 20.0,
        "Historical analytic nD=nT profile; gamma_n=0.657, gamma_T=0.75")
    add("earlier_30keV_ion_profile_replay", source_old30,
        2 * 1.2e20, 30.0,
        "Historical model-builder nD=nT profile; gamma_n=0.657, gamma_T=0.80")

    usable = 14.1 / 17.6 * CAPTURE_NEUTRON + 3.5 / 17.6 * DEPOSIT_ALPHA
    thresholds = {str(target): (target + RECIRC_GWE) /
                  (ACTIVE_CORES * usable * ETA) for target in (17.0, 22.0)}
    report = {
        "status": "POWER_BASIS_RECONCILIATION_NOT_NEW_PLANT_VALIDATION",
        "source_zip": source_path.name,
        "source_manifest_sha256_matches_three_MGRID_runs": manifest_hash,
        "source_members_sha256_verified": True,
        "bosch_hale_table_max_relative_error": reference_error,
        "density_m3_column": "n_D + n_T, verified from pressure=n_total*(Ti+Te)*keV_to_J",
        "pressure_identity_max_relative_error": pressure_error,
        "current_species_center_m3": density[0] / 2,
        "historical_model_builder_species_center_m3": 1.2e20,
        "density_scale_to_historical_species_center": density_scale,
        "fusion_source_scaling_at_fixed_profile": density_scale**2,
        "pressure_current_axis_Pa": pressure[0],
        "pressure_if_current_profile_density_is_rescaled_axis_Pa": pressure[0] * density_scale,
        "historical_17_GWe_planning_identity": "10*5.00 GWth captured/core*0.40 - 3.00 GWe = 17.00 GWe; a target, not a measured output",
        "historical_selected_reported": {
            "fusion_GWth_per_core": 5.4159,
            "captured_GWth_per_core": 5.2953,
            "gross_GWe": 20.44,
            "recirculating_GWe": 2.7219,
            "net_GWe": 17.718,
            "reference": "ARGONNE_PRELIMINARY_REPORT.md selected feasible_with_margin row",
        },
        "current_capture_screen_assumptions": {
            "neutron_capture": CAPTURE_NEUTRON,
            "alpha_deposition": DEPOSIT_ALPHA,
            "usable_fraction": usable,
            "cycle_efficiency": ETA,
            "recirculating_GWe": RECIRC_GWE,
            "unattributed_anchor_offset_applied": 0.0,
        },
        "threshold_fusion_source_GWth_per_core_with_current_capture_screen": thresholds,
        "scenarios": rows,
        "limits": "Changing the plasma density or temperature profile does not alter this vacuum MGRID. "
                  "All electric outputs in the scenario table are conditional arithmetic with shared capture/cycle/load inputs. "
                  "The higher-density scenario is not a measured operating point."
    }

    output_dir = Path(os.environ.get("TOMAHAWK_POWER_OUTPUT_DIR", "/kaggle/working" if
                       Path("/kaggle/working").is_dir() else str(Path.cwd() / "tomahawk_power_output")))
    output_dir.mkdir(parents=True, exist_ok=True)
    json_path = output_dir / "Tomahawk_16T_Power_Reconciliation.json"
    csv_path = output_dir / "Tomahawk_16T_Power_Reconciliation.csv"
    json_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    with csv_path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    zip_path = output_dir / "Tomahawk_16T_Power_Reconciliation_Results.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(json_path, json_path.name)
        archive.write(csv_path, csv_path.name)
    print("Original source checked against MGRID manifest:", manifest_hash)
    print("CSV density means nD+nT; nD(0)=nT(0)=", f"{density[0]/2:.3e}")
    for row in rows:
        print(row["scenario"], "fusion", f"{row['fusion_source_GWth_per_core']:.3f}",
              "GWth/core; net screen", f"{row['conditional_net_GWe_no_anchor_offset']:.3f}", "GWe")
    print("17/22 GWe thresholds at the screen assumptions:", thresholds)
    print("DONE", zip_path)
    return report


reconciliation = main()
